# 12a: Basic Forecasting

In a previous lecture, we looked at monthly US nuclear generation and decomposed it into trend-cycle, seasonality, and a remainder. Today we will use that structure to predict observations we have not yet seen, beginning with a familiar linear regression.

## Sources and Adaptation Notes

Adapted closely from Allen B. Downey, [*Think Stats*, third edition, §12.3, “Prediction”](https://allendowney.github.io/ThinkStats/chap12.html#prediction), and its [Chapter 12 notebook](https://github.com/AllenDowney/ThinkStats/blob/v3/nb/chap12.ipynb). The component-forecast section retains his wording and calculation sequence wherever applicable, with trend-cycle terminology clarified. The opening regression and leakage discussion are course additions; scikit-learn replaces the original regression interface.

[FPP Pythonic §5.8](https://otexts.com/fpppy/05-toolbox.html#evaluating-point-forecast-accuracy) supports the distinction between historical fit and future forecast accuracy. Attribution and reuse terms appear at the end.

## The Previous Decomposition

Reload the same data used in 11a and reproduce its STL result.

These are monthly US nuclear-generation measurements from January 2001 through June 2024, measured in gigawatt-hours (GWh). The data come from the US Energy Information Administration (EIA), using the [snapshot accompanying Downey’s *Think Stats*](https://raw.githubusercontent.com/AllenDowney/ThinkStats/v3/data/Net_generation_for_all_sectors.csv). We use the same unchanged nuclear-generation extract as in 11a.

In [ ]:
from pathlib import Path
from urllib.request import urlopen

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.linear_model import LinearRegression
from statsmodels.tsa.seasonal import STL, seasonal_decompose
from statsmodels.tsa.forecasting.stl import STLForecast
from statsmodels.tsa.arima.model import ARIMA

# Use the unchanged local extract, or the already published 11a data in Colab.
data_candidates = [
    Path("data/us-nuclear-generation.csv"),
    Path("content/lectures/12a-basic-forecasting/data/us-nuclear-generation.csv"),
    Path("lectures/12a-basic-forecasting/data/us-nuclear-generation.csv"),
]
data_path = next((p for p in data_candidates if p.exists()), None)
if data_path is not None:
    frame = pd.read_csv(data_path)
else:
    data_url = (
        "https://raw.githubusercontent.com/olearydj/INSY7130/main/"
        "lectures/11a-time-series-and-decomposition/data/us-nuclear-generation.csv"
    )
    with urlopen(data_url, timeout=30) as response:
        frame = pd.read_csv(response)

frame["month"] = pd.to_datetime(frame["month"], format="%b %Y")
nuclear = frame.set_index("month")["generation_gwh"].rename("Nuclear generation")
assert nuclear.index.is_unique and nuclear.index.is_monotonic_increasing
assert nuclear.notna().all()
assert nuclear.index.equals(pd.date_range(nuclear.index.min(), nuclear.index.max(), freq="MS"))
nuclear = nuclear.asfreq("MS")

plt.rcParams.update({"figure.figsize": (10, 4), "font.size": 12,
                     "axes.spines.top": False, "axes.spines.right": False})
actual_options = dict(color="C0", lw=1.3)
trend_options = dict(color="C1", lw=2)
model_options = dict(color="C3", lw=2, ls="--")
pred_options = dict(color="C2", lw=1.8)

# Replace the book's small plotting helper with Matplotlib.
def finish_plot(title):
    plt.title(title)
    plt.ylabel("Generation (GWh)")
    plt.xlabel("Year")
    plt.legend()
    plt.tight_layout()
    plt.show()
    plt.close()

In [ ]:
previous_stl = STL(nuclear, period=12, seasonal=13, robust=True).fit()
fig = previous_stl.plot()
fig.set_size_inches(10, 8)
for ax, label in zip(fig.axes, ["Observed", "Trend-cycle", "Seasonal", "Remainder"]):
    ax.set_ylabel(f"{label}\n(GWh)")
fig.axes[0].set_title("US nuclear generation: the previous decomposition")
fig.tight_layout()
plt.show()
plt.close(fig)

The trend-cycle describes slower changes, the seasonal component describes recurring annual variation, and the remainder contains what those estimates leave unexplained. Forecasting requires a rule for extending the estimated structure beyond the observations available at the time.

## Training and Test Series

We can use the results from seasonal decomposition to predict future observations. To demonstrate, we'll use the following function to split the time series into a **training series**, which we'll use to generate predictions, and a **test series**, which we'll use to see whether they are accurate.

In [ ]:
def split_series(series, n=60):
    training = series.iloc[:-n]
    test = series.iloc[-n:]
    return training, test

With `n=60`, the duration of the test series is five years, starting in July 2019.

In [ ]:
training, test = split_series(nuclear)

# show the timestamp for the first observation of the test set
test.index[0]

Our objective is to use the training data to build a predictive model and use the test data to evaluate its performance. This is the classic **supervised learning** pattern.

Our training data ends June 2019, and test data begins the following month. We want to forecast (predict) the value of observations for July 2019 on, for the same dates as the test data.

Every model and decomposition below will be fit using `training` only.

In [ ]:
display(pd.DataFrame({
    "First month": [training.index.min(), test.index.min()],
    "Last month": [training.index.max(), test.index.max()],
    "Observations": [len(training), len(test)],
}, index=["Training", "Test"]))
assert training.index.max() < test.index.min()
assert len(test) == 60

## A Linear Regression Baseline

Use linear regression to fit a straight line to the observed training series using scikit-learn.

This baseline uses elapsed time as its only predictor. For now, we will ignore seasonal predictors and other patterns that we know contribute.

Count months from the beginning of the series. Fit the relationship between month number and observed generation, then extend the same time coordinate into the test period.

`LinearRegression` is scikit-learn’s ordinary least-squares regression estimator. `LinearRegression()` creates the model; `fit(X, y)` estimates the coefficients that minimize squared differences between observed and fitted values. Here `X` is a table with one predictor column, month number, and `y` is observed generation. After fitting, `predict(X)` returns predictions for the supplied month numbers.

In [ ]:
months = np.arange(len(nuclear))
X = pd.DataFrame({"months": months}, index=nuclear.index)
X_train = X.loc[training.index]
X_test = X.loc[test.index]

baseline = LinearRegression()
baseline.fit(X_train, training)
baseline_pred = pd.Series(baseline.predict(X), index=nuclear.index)
baseline_forecast = baseline_pred.loc[test.index]

display(pd.Series({"Intercept (GWh)": baseline.intercept_,
                   "Slope (GWh per month)": baseline.coef_[0]}).round(2))

`fit` estimates the intercept and slope from the training observations. `predict` applies those fitted coefficients to the supplied month numbers. The test months continue the original count as the only predictor.

In [ ]:
training.plot(label="training", **actual_options)
baseline_pred.plot(label="linear regression", **model_options)
plt.axvline(test.index[0], color="0.4", ls=":", label="test period begins")
finish_plot("Time-only regression and its forecast")

This result is a simple estimate of a linear trend. It captures the overall direction and rate of change across the training period, but it does not *isolate* that trend. Seasonal variation, longer cycles, and unusual observations can influence the fitted line.

The straight line cannot reproduce the annual peaks and troughs. Can a forecast that retains the seasonal pattern do better?

A **baseline** is a simple reference model against which we compare other approaches. This straight-line forecast establishes what we can achieve using only elapsed time. A more elaborate method should earn its additional complexity through a useful improvement on the same prediction task. We will compare both forecasts against the same withheld observations.

## Prediction With Decomposed Components

Now, suppose it's June 2019 and you are asked to generate a five-year forecast for electricity production from nuclear generators. To answer this question, we'll use the training data to make a model and then use the model to generate predictions. We'll start with a seasonal decomposition of the training data.

In 11a, we used statsmodels’ `seasonal_decompose` to estimate the trend-cycle, seasonal component, and remainder. The opening figure recalls our previous decomposition using STL. Here we return to the classical additive decomposition also used in 11a, with `period=12` for the annual cycle.

This time we pass only `training`, so the withheld observations cannot influence the estimated components.

In [ ]:
decomposition = seasonal_decompose(training, model="additive", period=12)
trend = decomposition.trend

Now we'll fit a linear model to the estimated trend-cycle. The explanatory variable, `months`, is the number of months from the beginning of the series.

The library calls this component `.trend`, and we keep `trend` as the variable name. Classical decomposition estimates trend and longer, nonseasonal cycles together. The regression fits a straight line through that combined estimate so we can extrapolate it.

In [ ]:
months = np.arange(len(trend))
data = pd.DataFrame({"trend": trend, "months": months}).dropna()
trend_model = LinearRegression()
trend_model.fit(data[["months"]], data["trend"])

Here is a summary of the results.

In [ ]:
display(pd.Series({
    "Intercept (GWh)": trend_model.intercept_,
    "Slope (GWh per month)": trend_model.coef_[0],
    "Training trend-cycle R²": trend_model.score(data[["months"]], data["trend"]),
}))

The $R^2$ value is about 0.35, which suggests that the straight line does not fit the estimated training trend-cycle particularly well. We can get a better sense of that by plotting the fitted line. We'll use the `predict` method to compute expected values for the training and test data.

In [ ]:
months = np.arange(len(training) + len(test))
df = pd.DataFrame({"months": months})
# predicting only on the month number
display(df)

pred_trend = pd.Series(trend_model.predict(df), index=nuclear.index)

Here's the estimated trend-cycle and the linear model.

In [ ]:
trend.plot(label="training trend-cycle", **trend_options)
pred_trend.plot(label="linear model", **model_options)
plt.axvline(test.index[0], color="0.4", ls=":", label="test period begins")
finish_plot("Estimated training trend-cycle and linear extrapolation")

There's a lot going on that's not captured by the linear model, but it looks like there is a generally increasing trend.

Next we'll use the seasonal component from the decomposition to compute a `Series` of monthly averages.

In [ ]:
seasonal = decomposition.seasonal
monthly_averages = seasonal.groupby(seasonal.index.month).mean()

We can predict the seasonal component by looking up the dates from the fitted line in `monthly_averages`.

In [ ]:
pred_seasonal = monthly_averages[pred_trend.index.month]
pred_seasonal.index = pred_trend.index

Finally, to generate predictions, we'll add the seasonal component to the trend.

In [ ]:
pred = pred_trend + pred_seasonal

Here's the training data and the predictions.

In [ ]:
pred.plot(label="prediction", **pred_options)
training.plot(label="training", **actual_options)
plt.axvline(test.index[0], color="0.4", ls=":", label="test period begins")
finish_plot("Trend plus repeated seasonal effects")

The predictions fit the training data reasonably well, and the forecast looks like a reasonable projection, based on the assumption that the long-term trend will continue.

Now, from the vantage point of the future, let's see how accurate this forecast turned out to be. Here are the predicted and actual values for the five-year interval from July 2019.

In [ ]:
forecast = pred[test.index]
forecast.plot(label="predicted", **pred_options)
test.plot(label="actual", **actual_options)
finish_plot("Component forecast versus withheld observations")

The first year of the forecast was pretty good, but production from nuclear reactors in 2020 was lower than expected -- possibly due to the COVID-19 pandemic -- and it never returned to the long-term trend.

To quantify the accuracy of the predictions, we'll use the mean absolute percentage error (MAPE), which the following function computes.

In [ ]:
def MAPE(predicted, actual):
    ape = np.abs(predicted - actual) / actual
    return np.mean(ape) * 100

MAPE(forecast, test)

In this example, the predictions are off by 3.81% on average.

The post-2019 departure is visible; its cause is not established by this calculation. The fitted trend and seasonal pattern were assumptions about what would persist. A plausible projection can miss a sustained change.

## A Packaged STL Forecast

Let's compare our explicit component forecast with another procedure using `STLForecast`. It estimates seasonality with STL, forecasts the seasonally adjusted series using a model we supply, and adds the seasonal forecast back automatically.

We supply `ARIMA` for the adjusted series, with the illustrative settings used in the statsmodels example. Consider this a preview: we will return to the model and explain its settings in the ARIMA lecture. These settings have not been selected by trying alternatives on our test observations.

In [ ]:
stl_forecaster = STLForecast(
    training,
    ARIMA,
    model_kwargs={"order": (1, 1, 0), "trend": "t"},
    period=12,
    seasonal=13,
    robust=True,
)
stl_result = stl_forecaster.fit()
stl_forecast = stl_result.forecast(steps=len(test))
assert stl_forecast.index.equals(test.index)
assert stl_forecast.notna().all()

stl_forecast.plot(label="STL + ARIMA forecast", color="C4", lw=1.8)
test.plot(label="actual", **actual_options)
finish_plot("Packaged forecast versus withheld observations")

The constructor specifies the decomposition and forecasting model, `fit()` estimates them from the training history, and `forecast()` returns predictions for the next 60 months. Here `seasonal=13` retains the seasonal smoother used in our earlier STL example; `period=12` specifies the annual cycle.

STL estimates the components; ARIMA supplies the rule for forecasting the adjusted series. `STLForecast` repeats the last estimated seasonal cycle and adds it to that forecast. This is an alternative forecasting procedure, not an equivalent packaged version of the preceding calculation: both decomposition and extrapolation change, so an accuracy difference cannot be attributed to STL alone.

## Compare Forecast Errors

Let's compare all three forecasts against the same 60 withheld observations:

- time-only regression
- classical decomposition with linear trend
- STL with a supplied ARIMA model

We will use the common definition of error: actual minus predicted generation. MAE averages the absolute errors; RMSE squares errors before averaging and taking a square root, so larger misses receive more weight. Both are measured in GWh.

In [ ]:
predictions = pd.DataFrame({
    "Time-only regression": baseline_forecast,
    "Classical + linear trend": forecast,
    "STL + ARIMA": stl_forecast,
})
assert predictions.index.equals(test.index)
assert predictions.notna().all().all()
errors = predictions.rsub(test, axis="index")
comparison = pd.DataFrame({
    "MAE (GWh)": errors.abs().mean(),
    "RMSE (GWh)": np.sqrt(errors.pow(2).mean()),
    "MAPE (%)": errors.abs().div(test, axis="index").mean() * 100,
})
display(comparison.round(2))

MAPE expresses absolute error relative to the actual value. It is usable for these positive generation values, but becomes undefined at zero and unstable near zero. The table summarizes one forecast origin and a five-year horizon; it does not establish which method is best at every origin or horizon.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True, sharey=True)
for ax, (name, values), color in zip(axes, predictions.items(), ["C3", "C2", "C4"]):
    ax.plot(test.index, test, label="actual", **actual_options)
    ax.plot(values.index, values, color=color, label=name, lw=1.8)
    ax.set_ylabel("GWh")
    ax.legend(loc="upper right")
axes[-1].set_xlabel("Year")
fig.suptitle("The same withheld observations, three forecasting procedures")
fig.tight_layout()
plt.show()
plt.close(fig)

Which procedure has the lowest MAE and RMSE? Does it capture the seasonal swings, the overall level, or both? Use the plots to explain an important limitation that the average error alone conceals.

## Accuracy Across the Forecast Period

The previous table averages errors across all five years. Let's see how accuracy changes over the forecast period. We will use the same predictions and calculate MAE and RMSE separately for the first year, the second year, and the remaining three years.

In [ ]:
forecast_windows = {
    "Months 1–12": slice(0, 12),
    "Months 13–24": slice(12, 24),
    "Months 25–60": slice(24, 60),
}
window_scores = []
for window, positions in forecast_windows.items():
    window_errors = errors.iloc[positions]
    scores = pd.DataFrame({
        "MAE (GWh)": window_errors.abs().mean(),
        "RMSE (GWh)": np.sqrt(window_errors.pow(2).mean()),
    })
    scores.index.name = "Method"
    scores["Forecast months"] = window
    window_scores.append(scores.reset_index())

horizon_comparison = pd.concat(window_scores, ignore_index=True)
display(horizon_comparison.set_index(["Forecast months", "Method"]).round(2))

Each group of rows summarizes a different part of the same forecast. The first year runs from July 2019 through June 2020; the second runs through June 2021. The last group covers July 2021 through June 2024.

Which method performs best in each period? Does the ranking change? Use the plots above to identify when the forecasts begin to miss the observed level.

As we move farther ahead, we also encounter different conditions in the observed series. A forecast made in June 2019 keeps using the pattern learned through that date. Repeating the analysis at later forecast dates would let the models learn from the more recent observations.

## Leakage

Be very careful to preserve the boundary between train and test data. Training a model on observations from dates you later plan to predict is a form of **leakage** because it *leaks* information about the future into the predictive model.

This form of leakage is rather obvious. Others are more subtle, covert, or even slightly sinister!

Leakage can also occur in project preprocessing, feature selection, or model settings if those stages use outcomes reserved for evaluation.

- **Preprocessing before splitting.** Estimate missing-value replacements, scaling parameters, or PCA directions using the full dataset, then evaluate on its held-out portion. The test observations have already influenced the representation.
- **Features that look ahead.** Calculate a centered rolling average or fill a gap using later observations, then use that value in a forecast supposedly made earlier.
- **Information recorded after the decision.** Use a final inspection result or a repair code to predict whether equipment will need attention before the inspection occurs. The field is present in the completed dataset but would not have been available at prediction time.
- **Overlapping observations across the split.** Put nearly identical sensor windows in training and test sets, then claim performance on new operating episodes. Much of the same signal appears on both sides. Choose the split to match the intended use; evaluating on new entities may also require keeping each entity’s records together.
- **Outcomes hidden in the inputs.** Include a service-escalation label, or a field that effectively encodes it, when clustering customer records, then use separation by escalation status as independent evidence that the clusters are meaningful. That comparison is circular because the outcome helped form the groups.

Describing a complete dataset is legitimate. The problem is using that description as evidence of prediction or independent validation when the information used to build the result already contains what we later claim to evaluate.

Fit data-dependent operations inside the available training history. If we repeatedly change these forecasts after inspecting this test period, we are using it for model development and need a separate final evaluation.

What would change if we forecast one year instead of five, or started at a different date? Repeating the comparison at successive forecast origins will let us investigate that next time.

## Reuse Notice

The adapted §12.3 text is © Allen B. Downey, *Think Stats*, third edition, used under [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/). Adapted narrative and new narrative in this notebook are distributed under the same license. Downey’s source code is available under the [MIT License](https://opensource.org/license/mit). Modifications include the opening regression baseline, scikit-learn regression calls, course plotting helpers, month-start labels, MAE/RMSE comparison, packaged STL+ARIMA forecast, trend-cycle terminology, and the leakage discussion. The component-forecast sequence retains Downey’s language closely. The nuclear-generation data are an unchanged extract of the EIA snapshot accompanying the book; see `data/README.md`.